# Task C — step 2: re-solve β* on P_θ draw A, evaluate on draw C

Finance side is `taskb` unchanged (same `g`, same Carr–Madan targets, same convex dual, same held-out sets). This notebook adds only the sample discipline of `DECISIONS.md` §5: β* and ψ̂ from draw A, every reported number and SE from draw C. Task B reference numbers are hard-coded from `taskb/full_run.log` for side-by-side.

Reports `beta_raw` (raw units) and screen margins in raw column-sd units. Results are logged in `DECISIONS.md` §10. Cell 0 is the Colab clone cell; locally, skip it.

In [ ]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch --all
!git checkout v2_code
!git status

In [ ]:
import os, sys, json, pickle, time
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
import taskc
from taskc.config import CFG
from taskc.ptheta import load_checkpoint, load_draw
from taskc.dual import solve_level, evaluate_on, baseline_on, TASKB, TASKB_EXOTICS_Q
from config import q_params, SimConfig, EXOTICS      # taskb
from heston import simulate                          # taskb
from projection import condition_number              # taskb

RUN_TAG = "lrema"                                    # the frozen P_theta (DECISIONS.md section 9)
RUN = replace(CFG, artifact_dir=CFG.run_dir(RUN_TAG))
_, std, cfg_dict, extra = load_checkpoint(os.path.join(RUN.artifact_dir, RUN.ckpt_name))
A, C = load_draw(RUN, "A"), load_draw(RUN, "C")
for nm, d in (("A", A), ("C", C)):
    print(f"draw {nm}: seed {d.seed} n={d.z.shape[0]:,} device {d.device} rejected {d.n_rejected}")
pA, pC = std.to_paths(A.z, "Ptheta_A"), std.to_paths(C.z, "Ptheta_C")
q = q_params()
pQ = simulate(q, SimConfig(n_paths=100_000, H=21, seed=CFG.train_seed + 500_000), "Q").without_variance()
LEVELS = ("C0", "C1", "C2", "C3")

## Baselines: P_θ unweighted (draw C) and the Heston-Q noise floor

In [ ]:
baseC, baseQ = baseline_on(pC, q), baseline_on(pQ, q)
print(f"P_theta draw C unweighted: held-out vanilla RMSE {baseC['ho_van_rmse']:.4f}   held-out martingale RMSE {baseC['ho_mart_rmse']:.5f}   max|mart profile| {baseC['mart_prof_max']:.4f}")
print(f"Heston-Q floor           : held-out vanilla RMSE {baseQ['ho_van_rmse']:.4f}   held-out martingale RMSE {baseQ['ho_mart_rmse']:.5f}")
print(f"Task B Heston-P unweighted: 0.1811 / 0.017931")

## Solve on A, evaluate on C, at C0–C3

In [ ]:
tilts, results = {}, {}
hdr = f"{'lvl':4s} {'m':>3s} {'scr':>4s} {'margin':>7s} {'first bite':>20s} {'|b_raw|':>8s} {'TaskB':>7s} {'ESS_A%':>7s} {'ESS_C%':>7s} {'TaskB':>6s} {'maxw_C':>7s} {'KL_A':>7s} {'E_C[L]':>7s} {'hoVan_C':>8s} {'hoVan_A':>8s} {'TaskB':>7s}"
print(hdr); print("-" * len(hdr))
for lvl in LEVELS:
    tilt, r, cs = solve_level(lvl, pA, q)
    scr = r.screen; j = int(np.argmin(scr["margin"]))
    eC, eA = evaluate_on(tilt, pC, q), evaluate_on(tilt, pA, q)
    tb = TASKB[lvl]
    print(f"{lvl:4s} {cs.m:3d} {'ok' if scr['all_ok'] else 'FAIL':>4s} {scr['margin'].min():7.3f} {cs.names[j]:>20s} {np.linalg.norm(r.beta_raw):8.2f} {tb['beta_raw']:7.2f} "
          f"{r.ess_frac*100:7.2f} {eC['ess_frac']*100:7.2f} {tb['ess_frac']*100:6.2f} {eC['max_weight_ratio']:7.1f} {r.kl:7.4f} {eC['E_L']:7.4f} "
          f"{eC['ho_van_rmse']:8.4f} {eA['ho_van_rmse']:8.4f} {tb['ho_van']:7.4f}   converged={r.converged} it={r.n_iter}")
    tilts[lvl] = tilt; results[lvl] = dict(solve=r, evalC=eC, evalA=eA, screen=scr, names=cs.names)
print("\nhoVan_A is the in-sample number (weights and held-out on the same draw) -- the like-for-like comparison with Task B's column.")
print("hoVan_C carries valid SEs; it also contains the level noise between two independent 1e5 draws (see DECISIONS.md section 10).")

## Feasibility screen at C3 (raw sd units) — expected to bite first at (21d, K=115) and the 10d strikes

In [ ]:
scr, names = results["C3"]["screen"], results["C3"]["names"]
for j in np.argsort(scr["margin"])[:8]:
    print(f"  {names[j]:24s} margin {scr['margin'][j]:6.3f} sd   position in [min,max] {scr['pos'][j]:.4f}")
plt.figure(figsize=(12, 3)); plt.bar(range(len(names)), scr["margin"]); plt.yscale("log"); plt.axhline(0.147, c="r", ls="--", lw=1, label="Task B C3 min margin")
plt.xticks(range(len(names)), names, rotation=90, fontsize=6); plt.ylabel("margin (sd)"); plt.legend(); plt.tight_layout(); plt.show()

## Held-out vanillas and exotics on draw C (weights from A)

In [ ]:
e = results["C3"]["evalC"]
print("held-out vanillas at C3 on C:")
for n_, px, c_, er, se in zip(e["hv_names"], e["hv_px"], e["hv_target"], e["hv_err"], e["hv_se"]):
    print(f"  {n_:22s} target {c_:8.4f}  Q* {px:8.4f}  err {er:+8.4f}  se {se:.4f}  ({er/se:+.1f} se)")
print("\nexotics (draw C, SE self-normalized on C):")
print(f"  {'':28s} {'P_theta unw':>16s} " + " ".join(f"{l:>16s}" for l in LEVELS) + f" {'Heston-Q':>9s}")
for k in EXOTICS:
    row = f"  {k:28s} {baseC['exotics'][k][0]:8.4f}+-{baseC['exotics'][k][1]:.4f}"
    for l in LEVELS:
        v, se = results[l]["evalC"]["exotics"][k]; row += f" {v:8.4f}+-{se:.4f}"
    print(row + f" {TASKB_EXOTICS_Q[k]:9.4f}")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for axi, k in zip(ax, EXOTICS):
    v = [baseC['exotics'][k][0]] + [results[l]['evalC']['exotics'][k][0] for l in LEVELS]
    s = [baseC['exotics'][k][1]] + [results[l]['evalC']['exotics'][k][1] for l in LEVELS]
    axi.errorbar(range(5), v, yerr=[2*x for x in s], marker="o", capsize=3, label="P_theta -> Q*")
    axi.axhline(TASKB_EXOTICS_Q[k], c="k", ls="--", lw=1, label="Heston-Q"); axi.set_xticks(range(5)); axi.set_xticklabels(["P"] + list(LEVELS)); axi.set_title(k); axi.legend(fontsize=7)
plt.tight_layout(); plt.show()

## Save tilts for step 3 (L* targets on draw B come from these, with ψ̂ from A only)

In [ ]:
os.makedirs(os.path.join(RUN.artifact_dir, "dual"), exist_ok=True)
pickle.dump(tilts, open(os.path.join(RUN.artifact_dir, "dual", "tilts_nb.pkl"), "wb"))
print("saved", os.path.join(RUN.artifact_dir, "dual", "tilts_nb.pkl"))